# 📚 네이버 데이터랩 도서 인기 검색어 트렌드 데이터 분석

네이버 데이터랩(통합 검색어 트렌드 API)을 통해 수집된 **도서 부문 인기 키워드 10개**의 기간별 검색 비율(`ratio`) 데이터를 Pandas DataFrame으로 로드하고 탐색 및 분석합니다.

---
### 📌 분석 대상 10개 도서 키워드
1. **안녕피터팬**
2. **베스트셀러**
3. **베스트셀러순위**
4. **수족관책**
5. **원소원정대**
6. **브레인악셀**
7. **옥스브리지의철학수업**
8. **니체의초월자**
9. **테오책**
10. **찰리멍거바이블**

## 1. 라이브러리 임포트 및 환경 설정

In [ ]:
import os
import glob
import pandas as pd

# 판다스 데이터프레임 출력 행/열 수 확대 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)
print('✅ Pandas 버전:', pd.__version__)

✅ Pandas 버전: 3.0.5


## 2. CSV 데이터 파일 로드
`data/` 디렉터리에 저장된 수집 결과 CSV 파일 중 가장 최근 파일을 자동으로 탐색하여 로드합니다.

In [ ]:
# data 폴더 내의 book_trend_*.csv 파일 자동 탐색
search_paths = [
    os.path.join('..', 'data', 'book_trend_*.csv'),
    os.path.join('data', 'book_trend_*.csv'),
    os.path.join(r'C:\projects\wepscraping-git\data', 'book_trend_*.csv')
]

found_files = []
for pattern in search_paths:
    found_files = glob.glob(pattern)
    if found_files:
        break

if not found_files:
    raise FileNotFoundError('❌ CSV 파일을 찾을 수 없습니다. search_trend.py를 먼저 실행해 주세요.')

# 가장 최근에 생성/수정된 파일 선택
latest_file = max(found_files, key=os.path.getmtime)
df = pd.read_csv(latest_file, encoding='utf-8-sig')

print(f'✅ 데이터 로드 성공: {latest_file}')
print(f'- 총 행 수: {df.shape[0]}행, 컬럼 수: {df.shape[1]}개')

✅ 데이터 로드 성공: data\book_trend_2026-09-01_2026-09-06.csv
- 총 행 수: 60행, 컬럼 수: 3개


## 3. 데이터프레임 기본 확인 (`head`, `info`, 고유값)

In [ ]:
# 상위 10개 행 미리보기
df.head(10)

,date,keyword,ratio
0,2026-09-01,니체의초월자,77.11978
1,2026-09-01,베스트셀러,49.14658
2,2026-09-01,베스트셀러순위,22.96686
3,2026-09-01,브레인악셀,85.46433
4,2026-09-01,수족관책,18.09738
5,2026-09-01,안녕피터팬,4.26706
6,2026-09-01,옥스브리지의철학수업,4.30686
7,2026-09-01,원소원정대,5.49698
8,2026-09-01,찰리멍거바이블,8.88290
9,2026-09-01,테오책,95.82772


In [ ]:
# 데이터프레임 기본 구조 및 결측치 확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date     60 non-null     str    
 1   keyword  60 non-null     str    
 2   ratio    60 non-null     float64
dtypes: float64(1), str(2)
memory usage: 1.5 KB


In [ ]:
# 수집된 키워드 목록 및 날짜 범위 확인
keywords = df['keyword'].unique()
start_date = df['date'].min()
end_date = df['date'].max()

print(f'📅 수집 기간: {start_date} ~ {end_date} (총 {df["date"].nunique()}일간)')
print(f'📚 수집 키워드 ({len(keywords)}개):')
for idx, kw in enumerate(keywords, start=1):
    print(f'  {idx:2d}. {kw}')

📅 수집 기간: 2026-09-01 ~ 2026-09-06 (총 6일간)
📚 수집 키워드 (10개):
   1. 니체의초월자
   2. 베스트셀러
   3. 베스트셀러순위
   4. 브레인악셀
   5. 수족관책
   6. 안녕피터팬
   7. 옥스브리지의철학수업
   8. 원소원정대
   9. 찰리멍거바이블
  10. 테오책


## 4. 키워드별 검색 비율 통계 요약 및 순위 비교 (`groupby`)
각 도서 키워드별로 평균 검색 비율(`mean`), 최대 검색 비율(`max`), 최소 검색 비율(`min`)을 집계하여 어떤 책이 가장 검색 인기가 높았는지 확인합니다.

In [ ]:
# 키워드별 검색 비율 통계 요약 (평균값 기준 내림차순 정렬)
summary_by_keyword = df.groupby('keyword')['ratio'].agg(
    평균검색비율='mean',
    최대검색비율='max',
    최소검색비율='min',
    표준편차='std'
).sort_values(by='평균검색비율', ascending=False)

summary_by_keyword.round(2)

,평균검색비율,최대검색비율,최소검색비율,표준편차
keyword,,,,
테오책,82.75,95.83,73.62,8.57
니체의초월자,77.70,100.00,62.31,13.29
브레인악셀,73.04,95.83,49.93,17.61
찰리멍거바이블,50.27,84.93,8.88,28.98
베스트셀러,41.53,49.15,31.00,7.27
수족관책,20.50,24.57,18.10,2.46
안녕피터팬,19.95,100.00,3.14,39.22
베스트셀러순위,19.47,22.97,15.46,2.95
옥스브리지의철학수업,12.88,37.82,4.31,12.57


## 5. 날짜별 키워드 피벗 테이블 (`pivot_table`)
날짜를 행(Row), 도서 키워드를 열(Column)로 배치하여 일자별 검색 흐름을 한눈에 비교할 수 있는 시계열 표를 생성합니다.

In [ ]:
# 피벗 테이블 생성
pivot_df = df.pivot_table(index='date', columns='keyword', values='ratio', fill_value=0)

# 최근 10일간의 검색 비율 확인
pivot_df.tail(10)

keyword,니체의초월자,베스트셀러,베스트셀러순위,브레인악셀,수족관책,안녕피터팬,옥스브리지의철학수업,원소원정대,찰리멍거바이블,테오책
date,,,,,,,,,,
2026-09-01,77.11978,49.14658,22.96686,85.46433,18.09738,4.26706,4.30686,5.49698,8.88290,95.82772
2026-09-02,84.92597,47.13855,21.71184,95.82772,18.17269,4.79417,9.42126,4.51807,80.08075,88.55989
2026-09-03,100.00000,44.72891,21.31024,78.46567,20.15562,3.89056,6.19111,6.47590,84.92597,84.11843
2026-09-04,72.40915,42.87148,18.14759,55.31628,20.05522,3.63955,6.72947,5.69779,54.50874,73.62045
2026-09-05,69.44818,30.99899,15.46184,49.93270,24.57329,3.13755,12.78600,9.53815,35.12786,74.42799
2026-09-06,62.31493,34.31224,17.19377,73.21668,21.96285,100.00000,37.81965,11.27008,38.08882,79.94616


## 6. 키워드별 최고 인기 일자 (Peak Day) 분석
각 도서 키워드가 기간 중 **어느 날짜에 가장 많은 검색(클릭/관심)을 기록했는지** 최대치 발생 날짜를 탐색합니다.

In [ ]:
# 각 키워드별 최대 검색 비율 발생 날짜와 값 추출
peak_records = []
for kw in df['keyword'].unique():
    sub_df = df[df['keyword'] == kw]
    max_row = sub_df.loc[sub_df['ratio'].idxmax()]
    peak_records.append({
        '키워드': kw,
        '최고인기일자': max_row['date'],
        '최대비율(ratio)': max_row['ratio']
    })

peak_df = pd.DataFrame(peak_records).sort_values(by='최대비율(ratio)', ascending=False).reset_index(drop=True)
peak_df

,키워드,최고인기일자,최대비율(ratio)
0,니체의초월자,2026-09-03,100.00000
1,안녕피터팬,2026-09-06,100.00000
2,테오책,2026-09-01,95.82772
3,브레인악셀,2026-09-02,95.82772
4,찰리멍거바이블,2026-09-03,84.92597
5,베스트셀러,2026-09-01,49.14658
6,옥스브리지의철학수업,2026-09-06,37.81965
7,수족관책,2026-09-05,24.57329
8,베스트셀러순위,2026-09-01,22.96686
9,원소원정대,2026-09-06,11.27008


## 7. 도서 키워드 간 상관관계 분석 (`corr`)
어떤 도서 키워드들이 서로 비슷한 검색 트렌드 패턴을 보이며 함께 검색량이 늘어났는지 상관계수를 계산합니다.

In [ ]:
# 피벗 테이블 기반 키워드 간 상관계수 계산
correlation_matrix = pivot_df.corr()
correlation_matrix.round(2)

keyword,니체의초월자,베스트셀러,베스트셀러순위,브레인악셀,수족관책,안녕피터팬,옥스브리지의철학수업,원소원정대,찰리멍거바이블,테오책
keyword,,,,,,,,,,
니체의초월자,1.00,0.62,0.66,0.48,-0.47,-0.56,-0.62,-0.62,0.68,0.40
베스트셀러,0.62,1.00,0.95,0.72,-0.97,-0.48,-0.64,-0.90,0.20,0.75
베스트셀러순위,0.66,0.95,1.00,0.85,-0.92,-0.37,-0.53,-0.77,0.16,0.89
브레인악셀,0.48,0.72,0.85,1.00,-0.79,0.02,-0.10,-0.49,0.25,0.87
수족관책,-0.47,-0.97,-0.92,-0.79,1.00,0.28,0.45,0.82,-0.18,-0.75
안녕피터팬,-0.56,-0.48,-0.37,0.02,0.28,1.00,0.97,0.75,-0.20,-0.15
옥스브리지의철학수업,-0.62,-0.64,-0.53,-0.10,0.45,0.97,1.00,0.84,-0.17,-0.29
원소원정대,-0.62,-0.90,-0.77,-0.49,0.82,0.75,0.84,1.00,-0.33,-0.49
찰리멍거바이블,0.68,0.20,0.16,0.25,-0.18,-0.20,-0.17,-0.33,1.00,-0.16


## 8. 분석 결론 및 시사점
- **평균 검색량 최상위 키워드**: 기간 내 꾸준히 높은 검색 관심도를 유지한 도서/검색어 파악
- **스파이크(급상승) 키워드**: 특정 일자에 급격히 검색 비율이 100에 도달하며 이슈가 된 도서 파악
- **키워드 간 연관성**: 일반 명사(예: '베스트셀러')와 개별 서적명 간의 검색 추이 상관관계 확인